# MiliPoint reproduction — training & paper comparison

This notebook reproduces the results of **["MiliPoint: A Point Cloud Dataset for mmWave Radar"](https://arxiv.org/abs/2309.13425)**
(Cui, Zhong, Wu, Shen, Dahnoun, Zhao — NeurIPS 2023 Datasets & Benchmarks), using the code from
[`yizzfz/MiliPoint`](https://github.com/yizzfz/MiliPoint) (mirrored at
[`KhalidMehebub/Milipoint-reproduction`](https://github.com/KhalidMehebub/Milipoint-reproduction)).

**Before running:**
- Settings → Accelerator → **GPU T4 x2** (or P100).
- Settings → Internet → **On** (needed to `pip install`, clone the repo, download the dataset from
  Google Drive, and fetch the paper PDF for comparison).

**What this notebook does:**
1. Installs the `mmrnet` package and its dependencies.
2. Downloads the raw radar point-cloud dataset from Google Drive (`MiliPoint_data.zip`).
3. Trains + evaluates each `(task, model)` combination from the paper:
   - Tasks: `mmr_kp` (18-keypoint pose estimation), `mmr_iden` (person identification), `mmr_act` (action classification)
   - Models: `mlp`, `dgcnn`, `pointnet` (PointNet++), `pointtransformer`, `pointmlp`
4. Collects the reproduced test metrics into a table.
5. Pulls the paper PDF and attempts to auto-extract its results table (Table 3) for a side-by-side comparison
   — with a manual fallback since PDF table extraction is not 100% reliable.

**Time budget:** the paper trains for up to 300 epochs per run. With 3 tasks × 5 models = 15 runs, a full
sweep can take many hours even on a T4. A `QUICK_TEST` toggle is provided below to first validate the whole
pipeline end-to-end with a handful of epochs before committing to a full, paper-matching run.


## 1. Environment check

In [ ]:
!nvidia-smi
import torch, sys
print("Python:", sys.version)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| CUDA:", torch.version.cuda)


## 2. Install dependencies

We install `torch_geometric` plus the compiled `torch-scatter` / `torch-sparse` / `torch-cluster` extensions
matched to Kaggle's pre-installed torch + CUDA build (rather than reinstalling torch itself, which risks
breaking the GPU setup). If a prebuilt wheel isn't available for the exact combo, pip will fall back to a
source build (slower, a few minutes, but works on Kaggle's CUDA toolkit).


In [ ]:
import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() and torch.version.cuda else 'cpu'
PYG_URL = f"https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html"
print("Using wheel index:", PYG_URL)

%pip install -q torch_geometric

%pip install -q torch-scatter torch-sparse torch-cluster -f "$PYG_URL" || \
    %pip install -q torch-scatter torch-sparse torch-cluster

%pip install -q "pytorch_lightning==1.9.3" toml gdown pandas tqdm matplotlib scikit-learn pdfplumber


## 3. Get the code

In [ ]:
import os

REPO_DIR = "/kaggle/working/Milipoint-reproduction"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 https://github.com/KhalidMehebub/Milipoint-reproduction "$REPO_DIR"

%cd $REPO_DIR
!pip install -q -e .

import mmrnet
print("mmrnet OK, imported from:", mmrnet.__file__)


## 4. Download the dataset

`MiliPoint_data.zip` is hosted on Google Drive (linked from the repo's `readme.md`). This step needs
internet access enabled for the notebook.


In [ ]:
import os, glob

DATA_RAW = os.path.join(REPO_DIR, "data", "raw")
GDRIVE_FILE_ID = "1rq8yyokrNhAGQryx7trpUqKenDnTI6Ky"  # from the repo's readme.md
ZIP_PATH = "/kaggle/working/MiliPoint_data.zip"

n_pkl = len(glob.glob(os.path.join(DATA_RAW, "*.pkl")))
if n_pkl == 0:
    if not os.path.exists(ZIP_PATH):
        !gdown --id $GDRIVE_FILE_ID -O "$ZIP_PATH"
    !unzip -q -o "$ZIP_PATH" -d "$DATA_RAW"
    n_pkl = len(glob.glob(os.path.join(DATA_RAW, "*.pkl")))

print(f"{n_pkl} .pkl files in {DATA_RAW}")
assert n_pkl > 0, "Dataset did not download/unzip correctly — check the Google Drive link is still valid."


## 5. Experiment grid

Config `.toml` files (under `configs/`) and CLI defaults are taken verbatim from the upstream repo:
- optimizer: `adam`, learning rate `1e-5`, weight decay `1e-5`, batch size `128` (CLI defaults in `mmrnet/cli.py`)
- data split 0.8 / 0.1 / 0.1, stack of 5 frames, per-data-point zero padding (from the `*_stack_5_point.toml` configs)
- `mmr_kp` uses 18 keypoints (`num_keypoints='high'`) to match the paper's headline table; the repo defaults to 9 (`'low'`) unless overridden.

Set `QUICK_TEST = True` first to confirm every `(task, model)` pair trains and evaluates end-to-end with a
tiny epoch budget, before switching to `False` for a full, paper-matching 300-epoch run.


In [ ]:
QUICK_TEST = True   # flip to False for the full paper-matching run

TASKS = {
    "mmr_kp":  {"config": "configs/keypoints/mmr_keypoints_stack_5_point.toml", "extra": {"dataset_num_keypoints": "high"}},
    "mmr_iden": {"config": "configs/iden/mmr_iden_stack_5_point.toml", "extra": {}},
    "mmr_act": {"config": "configs/action/mmr_action_stack_5_point.toml", "extra": {}},
}
MODELS = ["mlp", "dgcnn", "pointnet", "pointtransformer", "pointmlp"]

TRAIN_CONFIG = dict(
    optimizer="adam",
    learning_rate=1e-5,
    weight_decay=1e-5,
    batch_size=128,
    max_epochs=(3 if QUICK_TEST else 300),
    num_workers=2,
    seed=20,
)
print("Grid:", [(t, m) for t in TASKS for m in MODELS])
print("Train config:", TRAIN_CONFIG)


## 6. Train + evaluate helper

This calls the same `mmrnet.session.train.train` / dataset / model code the CLI (`./mm train ...` /
`./mm eval ...`) uses, but in-process so we can capture the returned test metrics directly instead of
scraping stdout.


In [ ]:
import gc, time, json, logging
import toml as toml_lib
import torch
import pytorch_lightning as pl

from mmrnet.dataset import get_dataset
from mmrnet.models import model_map
from mmrnet.session.train import train as mm_train
from mmrnet.session.wrapper import ModelWrapper

logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)

RESULTS_CSV = "/kaggle/working/reproduced_results.csv"


def build_dataset_config(task_cfg):
    with open(task_cfg["config"]) as f:
        cfg = toml_lib.load(f)
    cfg["num_keypoints"] = {"low": 9, "high": 18}[task_cfg["extra"].get("dataset_num_keypoints", "low")]
    return cfg


def run_one(task_name, model_name, save_name):
    task_cfg = TASKS[task_name]
    mmr_dataset_config = build_dataset_config(task_cfg)

    train_loader, val_loader, test_loader, info = get_dataset(
        name=task_name,
        batch_size=TRAIN_CONFIG["batch_size"],
        workers=TRAIN_CONFIG["num_workers"],
        mmr_dataset_config=mmr_dataset_config,
    )

    model = model_map[model_name](info=info)

    plt_trainer_args = {
        "max_epochs": TRAIN_CONFIG["max_epochs"],
        "devices": 1,
        "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
        "strategy": None,
        "fast_dev_run": False,
        "enable_progress_bar": False,
        "logger": False,
    }

    save_path = f"checkpoints/{save_name}"
    t0 = time.time()
    mm_train(
        model=model, train_loader=train_loader, val_loader=val_loader,
        optimizer=TRAIN_CONFIG["optimizer"], learning_rate=TRAIN_CONFIG["learning_rate"],
        weight_decay=TRAIN_CONFIG["weight_decay"], plt_trainer_args=plt_trainer_args,
        save_path=save_path,
    )
    train_seconds = time.time() - t0

    # reload best checkpoint and evaluate on the held-out test split, mirroring mmrnet/session/test.py
    eval_model = ModelWrapper(model_map[model_name](info=info))
    state_dict = torch.load(f"{save_path}/best.ckpt")["state_dict"]
    eval_model.load_state_dict(state_dict)
    eval_model.eval()

    test_trainer = pl.Trainer(
        devices=1, accelerator="gpu" if torch.cuda.is_available() else "cpu",
        enable_progress_bar=False, logger=False,
    )
    test_results = test_trainer.test(eval_model, test_loader, verbose=False)[0]

    del model, eval_model, train_loader, val_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()

    return {"task": task_name, "model": model_name, "train_seconds": round(train_seconds, 1), **test_results}


## 7. Run the sweep

Resumable: if `reproduced_results.csv` already has a row for a `(task, model)` pair, it's skipped, so a
Kaggle session timeout doesn't lose earlier progress — just re-run this cell after restarting.


In [ ]:
import pandas as pd
import os

if os.path.exists(RESULTS_CSV):
    results_df = pd.read_csv(RESULTS_CSV)
else:
    results_df = pd.DataFrame()

done = set(zip(results_df.get("task", []), results_df.get("model", [])))

for task_name in TASKS:
    for model_name in MODELS:
        if (task_name, model_name) in done:
            print(f"skip {task_name}/{model_name} (already have a result)")
            continue
        save_name = f"{task_name}_{model_name}"
        print(f"=== training {save_name} ===")
        try:
            row = run_one(task_name, model_name, save_name)
        except Exception as e:
            print(f"FAILED {save_name}: {e}")
            continue
        print(row)
        results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
        results_df.to_csv(RESULTS_CSV, index=False)

results_df


## 8. Reproduced results, tidied up

- Classification tasks (`mmr_iden`, `mmr_act`) report `test_acc` (Top-1) and `test_top3_acc`.
- The keypoint task (`mmr_kp`) reports `test_mle` (mean localization error). The dataset applies a `x100`
  scale transform to keypoint coordinates, so `test_mle` is already in the same units (cm) the paper reports.


In [ ]:
pretty = results_df.copy()
if "test_acc" in pretty:
    pretty["test_acc_%"] = (pretty["test_acc"] * 100).round(2)
if "test_top3_acc" in pretty:
    pretty["test_top3_acc_%"] = (pretty["test_top3_acc"] * 100).round(2)
if "test_mle" in pretty:
    pretty["test_mle_cm"] = pretty["test_mle"].round(2)
pretty.sort_values(["task", "model"])


## 9. Paper's reported numbers, for comparison

This cell tries to auto-download the arXiv PDF and pull out its results table (Table 3) with `pdfplumber`,
since automated table extraction can misalign columns. **Always cross-check the printed page text against
the actual table in the PDF before trusting the parsed numbers** — open the PDF link below and look at
Table 3 (identification / action accuracy, keypoint MLE) yourself if anything looks off.


In [ ]:
import urllib.request

ARXIV_PDF = "https://arxiv.org/pdf/2309.13425"
PDF_PATH = "/kaggle/working/milipoint_paper.pdf"

try:
    urllib.request.urlretrieve(ARXIV_PDF, PDF_PATH)
    print("Downloaded paper to", PDF_PATH)
except Exception as e:
    print("Could not download the paper automatically:", e)
    print(f"Open it manually: {ARXIV_PDF}")


In [ ]:
import pdfplumber

if os.path.exists(PDF_PATH):
    with pdfplumber.open(PDF_PATH) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text() or ""
            if "MLE" in text or ("Iden" in text and "Action" in text):
                print(f"--- page {i+1} (looks like the results table) ---")
                print(text)
                print()


Fill in `paper_results` below by reading Table 3 from the printed page(s) above (or the PDF directly).
A few figures from the paper are pre-filled as a starting point — **verify every value**, these were
sourced from a secondary summary, not a direct read of the table:

- 18-keypoint MLE (cm): PointMLP 14.11 ± 0.22, PointNet++ 14.94 ± 0.03, PointTransformer 17.03 ± 0.13, DGCNN 18.51 ± 0.03
- Identification: paper reports >75% Top-1 accuracy across all point-based models
- Action classification: paper reports the best Top-1 accuracy is below 40% — a much harder task


In [ ]:
# task, model -> paper's reported metric value (fill in / correct after reading Table 3)
paper_results = {
    ("mmr_kp", "pointmlp"):         {"metric": "mle_cm", "value": 14.11},
    ("mmr_kp", "pointnet"):         {"metric": "mle_cm", "value": 14.94},
    ("mmr_kp", "pointtransformer"): {"metric": "mle_cm", "value": 17.03},
    ("mmr_kp", "dgcnn"):            {"metric": "mle_cm", "value": 18.51},
    ("mmr_kp", "mlp"):              {"metric": "mle_cm", "value": None},

    ("mmr_iden", "mlp"):              {"metric": "acc_%", "value": None},
    ("mmr_iden", "dgcnn"):            {"metric": "acc_%", "value": None},
    ("mmr_iden", "pointnet"):         {"metric": "acc_%", "value": None},
    ("mmr_iden", "pointtransformer"): {"metric": "acc_%", "value": None},
    ("mmr_iden", "pointmlp"):         {"metric": "acc_%", "value": None},

    ("mmr_act", "mlp"):              {"metric": "acc_%", "value": None},
    ("mmr_act", "dgcnn"):            {"metric": "acc_%", "value": None},
    ("mmr_act", "pointnet"):         {"metric": "acc_%", "value": None},
    ("mmr_act", "pointtransformer"): {"metric": "acc_%", "value": None},
    ("mmr_act", "pointmlp"):         {"metric": "acc_%", "value": None},
}


## 10. Side-by-side comparison

In [ ]:
rows = []
for _, r in pretty.iterrows():
    key = (r["task"], r["model"])
    paper = paper_results.get(key, {"metric": None, "value": None})
    if paper["metric"] == "mle_cm":
        reproduced = r.get("test_mle_cm")
    elif paper["metric"] == "acc_%":
        reproduced = r.get("test_acc_%")
    else:
        reproduced = None
    rows.append({
        "task": r["task"], "model": r["model"], "metric": paper["metric"],
        "reproduced": reproduced, "paper": paper["value"],
        "delta": (None if reproduced is None or paper["value"] is None else round(reproduced - paper["value"], 2)),
    })

comparison_df = pd.DataFrame(rows).sort_values(["task", "model"])
comparison_df


## Notes on interpreting differences

- `QUICK_TEST = True` only trains for a few epochs — do **not** compare those numbers to the paper.
  Set `QUICK_TEST = False` and re-run section 7 once the pipeline is validated end-to-end.
- The paper reports results averaged over multiple seeded runs (mean ± std); this notebook runs each
  `(task, model)` combination once with `seed=20` (the repo's default config seed). Some spread versus the
  paper's mean is expected from run-to-run variance alone.
- Small hyperparameter or library-version drift (PyTorch/PyG versions, hardware, exact epoch count if a run
  hits Kaggle's session time limit) can also shift results — check `train_seconds` per row to see if a run
  was cut short.
